# Two-Stage BERTimbau Base — RECLin-PT (Colab T4)

Treina a arquitetura **Two-Stage** de extração de relação sobre as partições do SemClinBr com o **BERTimbau Base** (`neuralmind/bert-base-portuguese-cased`; a execução de referência usa `pucpr/biobertpt-all` em `reclin_two_stage.ipynb`, que **não** é alterado): dois classificadores binários independentes, cada um com o seu fine-tuning a partir dos pesos **originais** (nenhum estágio parte de checkpoint ajustado).

* **Estágio 1** — existe relação? `relacao` × `no_relation`, com todos os pares candidatos do TRAIN. 3 épocas; checkpoint escolhido pelo maior **AP de `relacao`** no DEV completo.
* **Estágio 2** — qual o tipo? `negation_of` × `associated_with`, só com os pares do TRAIN que têm relação anotada. 10 épocas; checkpoint escolhido pelo maior **F1 de `negation_of`** nos pares do DEV com relação anotada.
* **Composição (inferência)** — P(`relacao`) ≥ **τ** manda o par ao estágio 2; os demais ficam `no_relation`. **τ é escolhido no DEV**, pelo fluxo real completo, e congelado em `limiar.json` **antes** do TEST.

Mesma representação de entrada, seed, partições, parâmetros de treino e infraestrutura do Two-Stage com `biobertpt-all`; **só muda o encoder** (e, com ele, a revisão dos pesos, a identidade `config_id` e os nomes das pastas). O BERTimbau é uma **variante fora do pré-registro** da fase (que fixa `pucpr/biobertpt-all`): não recebe nome oficial e serve à comparação no DEV (seção 10b). Protocolo: `docs/protocolo/fase-biobertpt-seed42.md`. **Este notebook não avalia o TEST**: o TEST é avaliado uma única vez, em etapa separada (`two_stage.py avaliar-test`), com o τ já congelado.

Antes de rodar: **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)** e cadastre os secrets `GITHUB_PAT` e `HF_TOKEN` (ícone de chave 🔑, com **Notebook access** ligado).

Cada estágio treina no disco local e é **espelhado a cada checkpoint no Google Drive** (`MyDrive/RECLin-PT/execucoes/`); se o runtime cair, **rode de novo a seção do estágio**: ela **retoma automaticamente** da execução interrompida compatível.

Ordem: seção 7 (estágio 1) → seção 8 (estágio 2) → seção 9 (compor e escolher τ no DEV) → seção 10 (resultados). Os envios para fora (GitHub na seção 11, Hugging Face na seção 12) só acontecem depois de você mudar a confirmação explícita para `True`.

## 1. Clonar o repositório

In [ ]:
import os, shutil, subprocess, sys, tempfile
from pathlib import Path
from google.colab import userdata

try:
    userdata.get('GITHUB_PAT')               # secret do Colab, nunca hard-coded (o valor não é guardado nem impresso)
except Exception as erro:
    raise SystemExit('Secret GITHUB_PAT indisponível (%s). Colab: ícone de chave → "+ Add new secret" → nome '
                     'GITHUB_PAT → cole o token → ligue "Notebook access" → rode esta célula de novo.' % type(erro).__name__)
REPO_NAME = 'RECLin-PT'                       # nome do repo no GitHub
REPO = f'/content/{REPO_NAME}'
GIT_USER_NAME = 'angeloalsf'
GIT_USER_EMAIL = 'angeloalsf@gmail.com'

# O token NÃO entra na URL (ficaria gravado em .git/config): vai por GIT_ASKPASS (script 700, apagado ao
# terminar), só no ambiente do processo do git.
url = f'https://github.com/angeloalsf/{REPO_NAME}.git'

def git(*args, check=True):
    pasta = tempfile.mkdtemp(prefix='askpass-'); os.chmod(pasta, 0o700)
    script = os.path.join(pasta, 'askpass.sh')
    with open(script, 'w') as f:
        f.write('#!/bin/sh\ncase "$1" in *sername*) printf "x-access-token";; *) printf "%s" "$RECLIN_GIT_TOKEN";; esac\n')
    os.chmod(script, 0o700)
    env = {**os.environ, 'GIT_ASKPASS': script, 'GIT_TERMINAL_PROMPT': '0', 'RECLIN_GIT_TOKEN': userdata.get('GITHUB_PAT')}
    try:
        return subprocess.run(['git', '-c', 'credential.helper=', *args], env=env, check=check)
    finally:
        shutil.rmtree(pasta, ignore_errors=True)

if not os.path.isdir(REPO):
    git('clone', url, REPO)
else:
    print('Repo já clonado; fazendo pull --rebase para integrar mudanças remotas.')
    git('-C', REPO, 'pull', '--rebase', '--autostash', check=False)

git('-C', REPO, 'config', 'user.name', GIT_USER_NAME)
git('-C', REPO, 'config', 'user.email', GIT_USER_EMAIL)
git('-C', REPO, 'remote', 'set-url', 'origin', url)

# Código novo do projeto: fica em REPO/codigo. Registra o commit clonado (HEAD) na proveniência de cada execução.
BRANCH = subprocess.run(['git', '-C', REPO, 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO}/codigo')
from reclin.experimento import apresentacao, codigo_colab, operacoes, plano, raiz, segredos
INFO = codigo_colab.clonar_codigo(f'angeloalsf/{REPO_NAME}', BRANCH, REPO)
SHA_CODIGO = INFO['sha']
CODIGO = Path(INFO['codigo'])
raiz.ativar(CODIGO)                          # põe codigo/ no sys.path (devolve o ambiente de subprocessos, não o caminho)
print('Repo pronto em', REPO, f'| branch {BRANCH} | commit {SHA_CODIGO}')

## 2. Verificar a GPU (T4)

In [ ]:
!nvidia-smi
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')
assert torch.cuda.is_available(), 'Sem GPU! Ative em Ambiente de execução → GPU (T4).'

## 3. Montar o Google Drive (checkpoints)

Execuções em `MyDrive/RECLin-PT/execucoes/<nome>/` — **pasta própria** de cada execução (o nome inclui a configuração). Dentro: `checkpoints/melhor_modelo/` (pesos da melhor época) e `checkpoints/ultimo.pt` (estado de retomada). O treino roda no disco local e o Drive recebe uma cópia verificada (SHA-256).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RAIZ_DRIVE = Path(f'/content/drive/MyDrive/{REPO_NAME}')
RAIZ_DRIVE.mkdir(parents=True, exist_ok=True)
RAIZ_LOCAL = Path('/content/reclin_resultados/execucoes')       # disco local do Colab
RAIZ_PESOS = Path('/content/reclin_pesos')
print('Execuções espelhadas em:', RAIZ_DRIVE)
existe = any(e['estado'] in ('interrompida', 'iniciada') for e in plano.execucoes(RAIZ_LOCAL, RAIZ_DRIVE))
print('execução interrompida existente?', existe, '(se True, o treino vai RETOMAR de onde parou)')

## 4. Instalar dependências

**Só esta célula instala dependências** (nada é instalado ao abrir o notebook nem em outras seções). Fonte única: `codigo/requirements.txt` do repo — a mesma lista usada fora do Colab, com as versões **fixadas com `==`**.

O `pip` **não reinstala** o `torch` enquanto a imagem do Colab trouxer a versão fixada (`torch==2.11.0`; o `+cu128` da imagem é ignorado na comparação pela PEP 440), e o mesmo vale para os demais pins que já batem com a imagem, então esta célula é rápida.

⚠️ Isso vale **enquanto o Colab ficar no 2.11.0**. Se a imagem subir e esta célula começar a demorar (baixa o `torch` do PyPI, vários GB), recapture as versões e suba o pin, nunca volte para `>=`. Nunca escreva `torch==2.11.0+cu128`. Se o pip trocar um pacote já importado (por exemplo `numpy`), o Colab pede para **reiniciar a sessão**: reinicie e rode tudo de novo, do início.

In [ ]:
!pip install -q -r $REPO/codigo/requirements.txt

## 5. Conferir os splits e os pesos

As partições (`codigo/dados/particoes-v2/*.jsonl`, seed 42, nível de documento) são **versionadas e congeladas** no repo — **as mesmas para os dois modelos**. Esta célula só **confere** (arquivos e SHA-256 contra o MANIFEST); **não baixa nem reconstrói** dados.

**Só a segunda célula desta seção baixa os pesos** (nenhuma outra seção acessa o Hub para baixar; o treino recusa se eles faltarem). Os pesos do encoder são baixados do Hub na **revisão fixada** (`REVISAO_PESOS`), com conferência dos hashes dos arquivos; o commit usado fica registrado e congelado para a fase.

In [ ]:
ENCODER = 'bertimbau'                  # chave em reclin.config.ENCODERS
REPO_PESOS = 'neuralmind/bert-base-portuguese-cased'
NOME_PESOS = 'bert-base-portuguese-cased'   # pasta dos pesos: bert-base-portuguese-cased@<revisão> (nunca a de biobertpt-all)
REVISAO_PESOS = '94d69c95f98f7d5b2a8700c420230ae10def0baa'   # commit de neuralmind/bert-base-portuguese-cased (SHA congelado; confira no Hub antes do 1º treino)

def _espec(estagio):
    from reclin.estrategias import two_stage           # importadas na hora (depois da instalação da seção 4)
    from reclin.estrategias.two_stage import treino
    from reclin.config import ENCODERS
    assert ENCODERS[ENCODER] == REPO_PESOS, 'ENCODER e REPO_PESOS divergem'
    return operacoes.Especificacao(estrategia=two_stage.ESTRATEGIA_ESTAGIO[estagio], script='two_stage.py',
                                   config_cls=two_stage.CONFIGS[estagio],
                                   estrategia_config=treino.registro_estagio(estagio), estagio=estagio,
                                   encoder=ENCODER)

def _ctx(estagio=1, **kw):
    return apresentacao.contexto(CODIGO, RAIZ_LOCAL, RAIZ_DRIVE, RAIZ_PESOS, REVISAO_PESOS, _espec(estagio),
                                 repositorio_pesos=REPO_PESOS, nome_pesos=NOME_PESOS, **kw)

def _ctx_exec(estagio, restaurar=False, **kw):
    # contexto da execução mais recente do estágio (local ou Drive); restaurar=True traz do Drive para o disco local
    try:
        return _ctx(estagio, execucao=apresentacao.execucao_atual(_ctx(estagio), restaurar=restaurar), **kw)
    except operacoes.ErroOperacao as erro:
        raise SystemExit(f'ERRO: {erro}') from None

def _ctx_ou_none(estagio, **kw):
    # contexto da execução mais recente do estágio, ou None se ainda não há (as etapas se enviam sem esperar uma pela outra)
    try:
        return _ctx(estagio, execucao=apresentacao.execucao_atual(_ctx(estagio), restaurar=True), **kw)
    except operacoes.ErroOperacao as erro:
        print(f'Estágio {estagio}: {erro}')
        return None

def _natureza(ctx):
    reg = proveniencia.ler(apresentacao.pasta_local_da_execucao(ctx))
    base = ctx.espec.config_protocolo().como_dict()
    return 'PROTOCOLO' if all(reg['parametros_de_treinamento'].get(k) == base[k] for k in ctx.espec.permitidos) else 'EXPLORATÓRIA'

def _treinar(estagio, parametros, nova_execucao, motivo):
    # mesma lógica do baseline: treino novo, retomada ou (protocolo concluído) nada, sempre depois do quadro pré-treino
    ctx = _ctx(estagio, parametros=parametros, motivo=motivo.strip() or 'sem motivo informado')
    protocolo = operacoes.config_da_operacao(ctx.espec, parametros) == ctx.espec.config_protocolo()
    rec = apresentacao.recomendar_ramo(ctx)
    if nova_execucao and rec['recomendada'] == operacoes.OP_RETOMAR and not motivo.strip():
        raise SystemExit(f"Há execução interrompida COMPATÍVEL ({', '.join(rec['compativeis'])}). Para RETOMAR, ponha "
                         "NOVA_EXECUCAO = False. Para abandoná-la e iniciar OUTRA tentativa, preencha MOTIVO.")
    operacao = operacoes.OP_NOVA if nova_execucao else rec['recomendada']
    print('\nOperação:', operacao, '| natureza:',
          'PARÂMETROS DO PROTOCOLO, encoder bertimbau (variante fora do pré-registro; DEV apenas)' if protocolo
          else 'EXPLORATÓRIA (fora da fase científica, não elegível ao TEST)')
    if operacao == operacoes.OP_NOVA and not nova_execucao:
        print('\nNada foi treinado:', rec['motivo'])
        print('Para outra tentativa ou parâmetros diferentes do protocolo (exploratória), ponha NOVA_EXECUCAO = True.')
        return None
    quadro = apresentacao.quadro_pre_treino(ctx, operacao, sha_codigo=SHA_CODIGO, repositorio=f'angeloalsf/{REPO_NAME}')
    try:
        apresentacao.exigir_quadro(quadro, ctx, operacao)
    except operacoes.ErroOperacao as erro:
        raise SystemExit(f'ERRO: {erro}') from None
    return operacoes.executar(operacao, ctx)

splits = os.path.join(REPO, 'codigo', 'dados', 'particoes-v2')
need = ['train.jsonl', 'dev.jsonl', 'test.jsonl', 'MANIFEST.json']
have = all(os.path.isfile(os.path.join(splits, f)) for f in need)

if not have:
    raise FileNotFoundError('Partições não encontradas em codigo/dados/particoes-v2/. Elas são versionadas e '
                            'congeladas; este notebook não baixa nem reconstrói dados. Confira a branch clonada.')

from reclin.experimento import proveniencia

PART = apresentacao.conferir_particoes(_ctx(1), sha_codigo=SHA_CODIGO)
assert PART and PART['ok'], 'Partições não conferem com o MANIFEST (veja ERRO acima).'

In [ ]:
apresentacao.avisar_revisao_pesos(_ctx(1))
RES_PESOS = operacoes.executar(operacoes.OP_PESOS, _ctx(1))      # os dois estágios usam os mesmos pesos originais
assert RES_PESOS, 'Pesos indisponíveis (veja ERRO acima).'

## 6. Token do Hugging Face Hub (Colab Secrets)

Prepara o **backup** da execução num repositório **privado** do HF Hub (`angeloalsf/reclin-pt-twostage-bertimbau-seed42`). É camada **extra**: o Drive continua sendo o destino principal. O envio é feito **ao final**, na seção 12, e só com a sua confirmação explícita (o `two_stage.py` atual não envia nada durante o treino).

O token nunca é colado em texto plano aqui; vem dos **Secrets** do Colab, e **não é exportado** para o ambiente nesta célula.

1. Crie um Access Token do tipo **Write** em <https://huggingface.co/settings/tokens>
2. No Colab: ícone de **chave** no painel esquerdo → **+ Add new secret** → nome `HF_TOKEN` → cole o valor → ligue **Notebook access**
3. Rode a célula abaixo

Sem o secret o treino roda igual; só o backup extra fica de fora.

In [ ]:
from google.colab import userdata

HF_USER = 'angeloalsf'
MODELO = 'bertimbau'

# A semente é a da fase, vem da constante do protocolo (não de nome de pasta).
SEED, _ = apresentacao.semente_da_fase()
HF_BACKUP_REPO = f'{HF_USER}/reclin-pt-twostage-{MODELO}-seed{SEED}'

try:
    cred = segredos.obter(segredos.HUGGING_FACE)
except Exception as erro:
    cred = None
    print('Secret HF_TOKEN indisponivel (%s).' % type(erro).__name__)

HF_OK = False
if cred:
    try:
        quem = segredos.verificar_escrita_hf(cred)        # consulta somente leitura (whoami); nada é enviado
        HF_OK = True
        print('HF_TOKEN carregado dos Secrets do Colab; token de escrita valido (usuario %s).' % quem['usuario'])
        print('(o valor nunca e impresso, nem exportado para o ambiente nesta celula).')
        print('Backup (secao 12) ->', HF_BACKUP_REPO, '(repositorio PRIVADO)')
        print('Semente da fase:', SEED)
    except Exception as erro:
        print('HF_TOKEN presente, mas nao validou como token de escrita (%s).' % type(erro).__name__)
cred = None
if not HF_OK:
    print('BACKUP NO HF HUB DESLIGADO nesta execucao.')
    print('  1. Crie um token WRITE em https://huggingface.co/settings/tokens')
    print('  2. Colab: icone de chave (Secrets) no painel esquerdo >')
    print('     "+ Add new secret" > nome HF_TOKEN > cole o valor >')
    print('     ligue "Notebook access"')
    print('  3. Rode esta celula de novo.')
    print('  O treino roda normalmente; so o backup extra fica de fora.')

## 7. Treinar o estágio 1 — existe relação? (com retomada)

`relacao` × `no_relation`, 3 épocas, `class_weight=balanced` (calculado no TRAIN do estágio), marcadores `[E1] [/E1] [E2] [/E2]` e janelas iguais às do baseline. Melhor época pelo **AP de `relacao` no DEV completo**; o **test não é usado aqui**.

A célula escolhe sozinha o que fazer: **treino novo** se não há execução do protocolo, **retomada** se há uma interrompida compatível (pula as épocas já feitas) e, se o protocolo já foi concluído, **não treina de novo** (nada é sobrescrito). Se o runtime cair, **rode esta célula de novo**.

> Mudar qualquer valor de `PARAMETROS` torna a execução **exploratória** (fora da fase científica, não elegível ao TEST) e só ela roda com `NOVA_EXECUCAO = True`. **Os dois estágios precisam compartilhar** `max_gap`, `ctx_chars`, `max_length` e `batch_size` (a composição recusa o par se divergirem). Retomar só aceita execução **compatível** (mesma configuração, dados, pesos e identidade do código).

In [ ]:
PARAMETROS_E1 = {              # valores do protocolo para o estágio 1 (3 épocas)
    'max_gap': 25, 'ctx_chars': 128, 'max_length': 128, 'epochs': 3, 'batch_size': 64, 'lr': 2e-5,
    'weight_decay': 0.01, 'warmup_ratio': 0.1, 'max_grad_norm': 1.0, 'class_weight': 'balanced',
}
NOVA_EXECUCAO = False          # True só para outra tentativa ou parâmetros diferentes do protocolo (exploratória)
MOTIVO = ''                    # opcional (vazio grava "sem motivo informado")

RESULTADO_E1 = _treinar(1, PARAMETROS_E1, NOVA_EXECUCAO, MOTIVO)

## 8. Treinar o estágio 2 — qual o tipo? (com retomada)

`negation_of` × `associated_with`, só com os pares do TRAIN que têm relação anotada, **10 épocas**. Melhor época pelo **F1 de `negation_of` nos pares do DEV com relação anotada**. Parte dos pesos **originais** (não do estágio 1 nem do baseline); o estágio 2 treina com os rótulos anotados, nunca com predições do estágio 1.

O comportamento é o da seção 7 (novo, retomada ou nada). Rode esta seção **depois** do estágio 1 concluído ou em outra sessão: as duas execuções são independentes.

In [ ]:
PARAMETROS_E2 = {              # valores do protocolo para o estágio 2 (10 épocas)
    'max_gap': 25, 'ctx_chars': 128, 'max_length': 128, 'epochs': 10, 'batch_size': 64, 'lr': 2e-5,
    'weight_decay': 0.01, 'warmup_ratio': 0.1, 'max_grad_norm': 1.0, 'class_weight': 'balanced',
}
NOVA_EXECUCAO_E2 = False
MOTIVO_E2 = ''

RESULTADO_E2 = _treinar(2, PARAMETROS_E2, NOVA_EXECUCAO_E2, MOTIVO_E2)

## 9. Compor os estágios e escolher τ no DEV

Com os dois estágios **concluídos e do protocolo**, roda `scripts/two_stage.py compor-dev`: para cada τ da grade pré-registrada (0,05 a 0,95), o pipeline completo roda pelo fluxo real no **DEV** (o estágio 2 só sobre os pares roteados por aquele τ); τ é o de maior F1 de `negation_of`, com os desempates do protocolo, e fica **congelado em `limiar.json`**. O DEV final é recalculado pelo mesmo fluxo e precisa coincidir com a linha do τ escolhido. **O TEST não é lido.**

⚠️ Congelar τ **não se desfaz**: se a composição `twostage_bertimbau_seed42` já existe, nada é sobrescrito. Por isso a célula primeiro mostra os dois estágios e **só compõe com `CONFIRMO_COMPOR = True`**. Deixe `ESTAGIO1` e `ESTAGIO2` vazios para usar a execução mais recente de cada estágio.

In [ ]:
CONFIRMO_COMPOR = False             # True = confirmo compor estes dois estágios e congelar τ no DEV
ESTAGIO1 = ''                       # nome da pasta do estágio 1 (vazio = a execução mais recente)
ESTAGIO2 = ''                       # nome da pasta do estágio 2 (vazio = a execução mais recente)

C1 = _ctx_exec(1, restaurar=True, **({'execucao': ESTAGIO1} if ESTAGIO1 else {}))
C2 = _ctx_exec(2, restaurar=True, **({'execucao': ESTAGIO2} if ESTAGIO2 else {}))
NAT1, NAT2 = _natureza(C1), _natureza(C2)
print(f'Estágio 1: {C1.execucao} | natureza: {NAT1}')
print(f'Estágio 2: {C2.execucao} | natureza: {NAT2}')
if NAT1 != 'PROTOCOLO' or NAT2 != 'PROTOCOLO':
    raise SystemExit('Só estágios do PROTOCOLO podem ser compostos para a fase científica (há execução exploratória).')
if not CONFIRMO_COMPOR:
    print('\nNada foi composto. Para compor e congelar τ no DEV, ponha CONFIRMO_COMPOR = True e rode de novo.')
else:
    COMPOSICAO = operacoes.executar(operacoes.OP_COMPOR, _ctx(1, estagio1=C1.execucao, estagio2=C2.execucao))

## 10. Resultados e curvas (só DEV)

Histórico por época e métricas do DEV de **cada estágio**, e a escolha de τ da composição (tabela do `limiar.json` e a curva F1 × τ). As métricas finais da composição são as do **fluxo real** nos 3 rótulos da tarefa, comparáveis às do baseline. O TEST não é lido nem avaliado aqui.

In [ ]:
import json
import matplotlib.pyplot as plt

C1 = _ctx_exec(1, restaurar=True, **({'execucao': ESTAGIO1} if ESTAGIO1 else {}))
C2 = _ctx_exec(2, restaurar=True, **({'execucao': ESTAGIO2} if ESTAGIO2 else {}))
EXECUCAO_E1, EXECUCAO_E2 = C1.execucao, C2.execucao
for _c in (C1, C2):
    operacoes.executar(operacoes.OP_DEV, _c)

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for linha, _c in enumerate((C1, C2)):
    r = json.load(open(apresentacao.pasta_local_da_execucao(_c) / 'treino.json', encoding='utf-8'))
    h = r['dev_history']; ep = [e['epoch'] for e in h]; campo = r['campo_selecao']
    ax[linha][0].plot(ep, [e['train_loss'] for e in h], 'o-', label='train_loss')
    ax[linha][0].plot(ep, [e.get('dev_loss') for e in h], 's-', label='dev_loss')
    ax[linha][0].set_title(f'Estágio {linha + 1}: loss'); ax[linha][0].legend()
    ax[linha][1].plot(ep, [e.get(campo) for e in h], 'o-', color='tab:green')
    ax[linha][1].axvline(r['melhor_epoca'], color='gray', linestyle='--', linewidth=1)
    ax[linha][1].set_title(f'Estágio {linha + 1}: {campo} (melhor época {r["melhor_epoca"]})')
    for a in ax[linha]:
        a.set_xlabel('época'); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

from reclin.experimento import persistencia
NOME_COMPOSICAO = f'twostage_{ENCODER}_seed{operacoes.SEMENTE_DA_FASE}'
COMP = RAIZ_LOCAL / NOME_COMPOSICAO
if not COMP.is_dir() and (persistencia.raiz_das_execucoes(RAIZ_DRIVE) / NOME_COMPOSICAO).is_dir():
    persistencia.restaurar(RAIZ_DRIVE, NOME_COMPOSICAO, RAIZ_LOCAL, progresso=print)       # sessão nova: traz do Drive
if not (COMP / 'limiar.json').is_file():
    print('\nComposição ainda não feita (seção 9): sem τ nem métricas do fluxo completo.')
else:
    lim = json.load(open(COMP / 'limiar.json', encoding='utf-8'))
    met = json.load(open(COMP / 'metricas.json', encoding='utf-8'))
    print(f"\nτ congelado: {lim['limiar']} | regra: {lim['configuracao_busca'].get('regra', '—')}")
    dev = met['dev']
    print('DEV (fluxo real):', {k: dev[k] for k in ('macro_f1', 'micro_f1', 'weighted_f1', 'mcc', 'f1_per_class') if k in dev})
    print('Diagnóstico por estágio:', met.get('dev_diagnostico_estagios'))
    tab = lim['tabela']
    plt.figure(figsize=(6, 4))
    plt.plot([t['limiar'] for t in tab], [t['f1_negation_of'] for t in tab], 'o-', label='F1 negation_of')
    plt.plot([t['limiar'] for t in tab], [t['macro_f1'] for t in tab], 's-', label='macro-F1')
    plt.axvline(lim['limiar'], color='gray', linestyle='--'); plt.xlabel('τ'); plt.title('Escolha de τ no DEV')
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 10b. Comparação no DEV: `bertimbau` × `biobertpt-all`

Só **leitura** de `predicoes_dev.json` já gravados (sem treino, sem inferência, **sem TEST**). A comparação é sobre o **mesmo universo de candidatos do DEV** e é recusada, com a razão, se `y_true`/`labels` diferirem. A execução com `biobertpt-all` (`twostage_biobertpt_seed42`) precisa existir no disco local ou no Drive; se não existir, aparece como ausente e nada é inventado. τ e checkpoints são os escolhidos **no DEV** de cada variante.

In [ ]:
import json
from reclin.experimento import comparacao_dev, persistencia

NOME_REFERENCIA = f'twostage_{operacoes.ENCODER_DA_FASE}_seed{operacoes.SEMENTE_DA_FASE}'   # biobertpt-all
RAIZES = [RAIZ_LOCAL, persistencia.raiz_das_execucoes(RAIZ_DRIVE)]
try:
    cmp_dev = comparacao_dev.carregar_e_comparar(RAIZES, [NOME_REFERENCIA, NOME_COMPOSICAO])
    print(comparacao_dev.formatar(cmp_dev))
except comparacao_dev.ErroComparacao as erro:
    print('Comparação recusada:', erro)

def _ler(nome, arquivo):
    for raiz in RAIZES:
        caminho = raiz / nome / arquivo
        if caminho.is_file():
            return json.load(open(caminho, encoding='utf-8'))
    return None

for rotulo, nome in (('biobertpt-all', NOME_REFERENCIA), ('bertimbau', NOME_COMPOSICAO)):
    lim, met = _ler(nome, 'limiar.json'), _ler(nome, 'metricas.json')
    print(f'\n== {rotulo} ({nome})')
    if lim is None:
        print('  limiar.json ausente (composição não feita/encontrada).'); continue
    print('  τ escolhido no DEV:', lim.get('limiar'), '| regra:', lim.get('configuracao_busca', {}).get('regra', '—'))
    print('  diagnóstico por estágio:', (met or {}).get('dev_diagnostico_estagios'))
    print('  estágios usados:', {k: (met or {}).get(k) for k in ('estagio1', 'estagio2')})


## 11. Salvar resultados no GitHub (confirmação explícita)

Versiona, **só com os arquivos que as pastas já têm**, em `codigo/resultados/execucoes/<nome>/` de uma **branch própria** (`main` e `master` são recusadas; o merge para a main é manual, fora do notebook):

* **cada estágio:** `config.json` e `treino.json` (histórico de treino por época);
* **a composição:** `limiar.json` (τ congelado e a tabela completa da busca de τ), `metricas.json` (**só DEV**, do pipeline completo) e `predicoes_dev.json` (rótulos, predições e probabilidades do DEV).

Nunca vão: `ultimo.pt`, checkpoints, pesos, logs nem artefatos do TEST. O envio **recusa** execução não concluída, arquivo com segredo, `metricas.json` com a chave `test`, texto clínico (qualquer string longa) e composição de validação técnica. Cada item é conferido e enviado separadamente: um estágio pode ser salvo sem esperar o outro.

**Nada é enviado até você mudar `PUBLICAR_NO_GITHUB` para `True`.** Antes, a célula mostra exatamente o que seria enviado.

In [ ]:
from reclin.experimento import persistencia, versionamento

PUBLICAR_NO_GITHUB = False          # True = confirmo enviar estes arquivos do DEV ao GitHub
BRANCH_RESULTADOS = 'resultados-colab'   # fixa (main e master são recusadas)
NOME_COMPOSICAO = f'twostage_{ENCODER}_seed{operacoes.SEMENTE_DA_FASE}'

ALVOS = []                          # (rótulo, perfil, pasta)
for _n in (1, 2):
    _c = _ctx_ou_none(_n)
    if _c is not None:
        ALVOS.append((f'Estágio {_n}', 'estagio', apresentacao.pasta_local_da_execucao(_c)))
if not (RAIZ_LOCAL / NOME_COMPOSICAO).is_dir() and (persistencia.raiz_das_execucoes(RAIZ_DRIVE) / NOME_COMPOSICAO).is_dir():
    persistencia.restaurar(RAIZ_DRIVE, NOME_COMPOSICAO, RAIZ_LOCAL, progresso=print)
if (RAIZ_LOCAL / NOME_COMPOSICAO).is_dir():
    ALVOS.append(('Composição', 'composicao', RAIZ_LOCAL / NOME_COMPOSICAO))
else:
    print('Composição ainda não feita (seção 9).')

RES_GIT = {}
for rotulo, perfil, pasta in ALVOS:
    print(f'\n##### {rotulo}: {pasta.name}')
    _cred = None
    try:
        plano_git = versionamento.conferir(pasta, repositorio=f'angeloalsf/{REPO_NAME}', branch=BRANCH_RESULTADOS,
                                           sha_codigo=SHA_CODIGO, perfil=perfil)
        if PUBLICAR_NO_GITHUB and not plano_git['pode_salvar']:
            print('Não enviado: há PROBLEMA acima.')
        elif PUBLICAR_NO_GITHUB:
            _cred = segredos.Credencial('GITHUB_PAT', userdata.get('GITHUB_PAT'), 'Colab Secrets')
            RES_GIT[pasta.name] = versionamento.salvar(
                pasta, repositorio=f'angeloalsf/{REPO_NAME}', branch=BRANCH_RESULTADOS, sha_codigo=SHA_CODIGO,
                clone='/content/reclin_versionamento', nome_git=GIT_USER_NAME, email_git=GIT_USER_EMAIL,
                credencial=_cred, perfil=perfil)
            print('Push concluído (se houve mudanças).')
    except versionamento.ErroVersionamento as erro:
        print('ERRO:', segredos.limpar(str(erro), _cred))
    finally:
        _cred = None
if not PUBLICAR_NO_GITHUB:
    print('\nNada foi enviado. Confira acima; para enviar, ponha PUBLICAR_NO_GITHUB = True e rode de novo.')

## 12. Backup privado dos modelos dos estágios no HF Hub (confirmação explícita)

Publica o `melhor_modelo/` de **cada estágio** (com o manifesto de SHA-256, **sem** `ultimo.pt`) no repositório **privado** `HF_BACKUP_REPO` (seção 6), cada um na sua pasta: o estágio 1 em **`etapa1/`** e o estágio 2 em **`etapa2/`**. Cada etapa se publica **assim que concluir**, sem esperar a outra (flags separadas). O envio é um commit só de adições dentro da própria pasta: **nunca apaga nem sobrescreve** a outra etapa, e uma pasta que já existe no repositório é recusada. A composição **não** vai ao HF (só ao Drive e ao GitHub).

A célula primeiro mostra o plano (o que seria enviado) e **só envia** o estágio cuja flag for `True`. O `HF_TOKEN` é lido dos Secrets **só durante o envio** e removido do ambiente ao terminar.

In [ ]:
PUBLICAR_ETAPA1 = False             # True = confirmo publicar o estágio 1 em etapa1/ (repositório PRIVADO HF_BACKUP_REPO)
PUBLICAR_ETAPA2 = False             # True = confirmo publicar o estágio 2 em etapa2/

RES_PUBLICACAO = {}
for _n, _publicar in ((1, PUBLICAR_ETAPA1), (2, PUBLICAR_ETAPA2)):
    _c = _ctx_ou_none(_n, repositorio_hf=HF_BACKUP_REPO)
    if _c is None:
        continue
    _c.confirmacao = operacoes.frase_de_publicacao(_c.execucao, HF_BACKUP_REPO) if _publicar else ''
    print(f'\nEstágio {_n}: {_c.execucao} | natureza: {_natureza(_c)} | destino: etapa{_n}/')
    operacoes.executar(operacoes.OP_PUBLICACAO, _c)             # plano: nada é enviado
    if not _publicar:
        print(f'Nada foi enviado do estágio {_n}. Para publicar, ponha PUBLICAR_ETAPA{_n} = True e rode de novo.')
        continue
    try:
        with apresentacao.segredo_no_ambiente(segredos.HUGGING_FACE):
            RES_PUBLICACAO[_n] = operacoes.executar(operacoes.OP_PUBLICAR, _c)
        print(f'etapa{_n}/ em https://huggingface.co/' + HF_BACKUP_REPO)
    except segredos.ErroSegredo as erro:
        print('ERRO:', erro)
        print(apresentacao.instrucoes_secret(segredos.HUGGING_FACE))